# Batch Processing Demo

This notebook demonstrates batch processing for large datasets with:
- Memory usage monitoring
- Data type optimization
- Controlled differences for validation
- Progress tracking

## Setup

In [1]:
import polars as pl
import numpy as np
from pathlib import Path
from datetime import datetime
from polars_proc_compare.batch_utils import compare_in_batches, get_memory_usage

# Create working directories
work_dir = Path("workspace/batch_demo")
work_dir.mkdir(parents=True, exist_ok=True)

data_dir = work_dir / "data"  # For parquet files
data_dir.mkdir(exist_ok=True)

temp_dir = work_dir / "temp"  # For temporary files
temp_dir.mkdir(exist_ok=True)

output_dir = work_dir / "output"  # For results
output_dir.mkdir(exist_ok=True)

## Memory Monitoring

Track memory usage throughout the process:

In [2]:
def show_memory():
    """Display current memory usage."""
    mem = get_memory_usage()
    print(f"Used: {mem['used']:.1f} MB")
    print(f"Available: {mem['available']:.1f} MB")
    print(f"Percent: {mem['percent']:.1f}%")

print("Initial memory state:")
show_memory()

Initial memory state:
Used: 10089.4 MB
Available: 29736.6 MB
Percent: 25.3%


## Data Generation

Create sample datasets with controlled differences:

In [3]:
def create_sample_data(n_rows: int, n_cols: int, seed: int = 42) -> pl.DataFrame:
    """Create sample data with mixed types.
    
    Args:
        n_rows: Number of rows
        n_cols: Number of columns (excluding id and timestamp)
        seed: Random seed for reproducibility
    """
    np.random.seed(seed)
    
    # Create base data
    data = {
        "id": pl.Series(range(n_rows)),
        "timestamp": pl.Series(
            [datetime(2024, 1, 1).timestamp() + i * 86400 for i in range(n_rows)]
        ).cast(pl.Datetime)
    }
    
    # Add columns with different types
    for i in range(n_cols):
        if i % 3 == 0:  # Integer columns
            data[f"int_col_{i}"] = pl.Series(
                np.random.randint(-10000, 10000, n_rows)
            )
        elif i % 3 == 1:  # Float columns
            data[f"float_col_{i}"] = pl.Series(
                np.random.normal(0, 100, n_rows)
            )
        else:  # String columns
            categories = [f"cat_{j}" for j in range(10)]
            data[f"cat_col_{i}"] = pl.Series(
                np.random.choice(categories, n_rows)
            )
    
    return pl.DataFrame(data)

def create_modified_data(df: pl.DataFrame, 
                        col_mod_rate: float = 0.1, 
                        row_mod_rate: float = 0.01) -> pl.DataFrame:
    """Create comparison data with controlled differences.
    
    Args:
        df: Base DataFrame
        col_mod_rate: Percentage of columns to modify
        row_mod_rate: Percentage of rows to modify in each affected column
    """
    compare_df = df.clone()
    
    # Modify selected columns
    for col in df.columns[2:]:  # Skip id and timestamp
        if np.random.random() < col_mod_rate:
            rows_to_modify = np.random.choice(
                range(len(compare_df)), 
                size=int(len(compare_df) * row_mod_rate),
                replace=False
            )
            if col.startswith('int'):
                compare_df = compare_df.with_columns([
                    pl.when(pl.col('id').is_in(rows_to_modify))
                    .then(pl.col(col) + np.random.randint(100, 1000))
                    .otherwise(pl.col(col))
                    .alias(col)
                ])
            elif col.startswith('float'):
                compare_df = compare_df.with_columns([
                    pl.when(pl.col('id').is_in(rows_to_modify))
                    .then(pl.col(col) * np.random.uniform(1.1, 2.0))
                    .otherwise(pl.col(col))
                    .alias(col)
                ])
            else:  # String columns
                compare_df = compare_df.with_columns([
                    pl.when(pl.col('id').is_in(rows_to_modify))
                    .then(pl.lit('modified'))
                    .otherwise(pl.col(col))
                    .alias(col)
                ])
    
    return compare_df

# Create datasets
print("Creating base dataset...")
base_df = create_sample_data(n_rows=10_000, n_cols=98)  # 98 + id + timestamp = 100 columns
base_path = data_dir / "base.parquet"
base_df.write_parquet(base_path)
print(f"Base dataset shape: {base_df.shape}")

print("\nCreating comparison dataset...")
compare_df = create_modified_data(base_df)
compare_path = data_dir / "compare.parquet"
compare_df.write_parquet(compare_path)
print(f"Compare dataset shape: {compare_df.shape}")

print("\nMemory after data creation:")
show_memory()

Creating base dataset...
Base dataset shape: (10000, 100)

Creating comparison dataset...
Compare dataset shape: (10000, 100)

Memory after data creation:
Used: 10133.4 MB
Available: 29692.7 MB
Percent: 25.4%


## Batch Comparison

Run comparison with memory-efficient settings:

In [4]:
# Configure batch processing
config = {
    "base_path": str(base_path.absolute()),
    "compare_path": str(compare_path.absolute()),
    "key_columns": ["id"],
    "batch_size": 10,           # Process 10 columns at a time
    "chunk_size": 2_000,        # Process 2k rows at a time
    "n_workers": 1,             # Single worker to minimize memory
    "max_memory_usage": 512,    # 512MB memory limit per batch
    "max_memory_percent": 75.0, # Run GC at 75% memory usage
    "monitor_memory": True,     # Enable memory monitoring
    "verbose": True,           # Show progress messages
    "temp_dir": temp_dir.absolute()  # Use specific temp directory
}

print("Starting batch comparison...")
results = compare_in_batches(**config)

# Save and display results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
html_path = output_dir / f"comparison_{timestamp}.html"
results.to_html(html_path)
print(f"\nReport saved to: {html_path}")

print("\nComparison Summary:")
print(f"Total differences: {results.total_differences}")
print(f"Columns with differences: {len(results.comparison_results)}")

print("\nFinal memory state:")
show_memory()

Starting batch comparison...
Processing batch 1/10 (10 columns)

Original schema:
__row_id: UInt32
id: Int64
timestamp: Datetime(time_unit='us', time_zone=None)
int_col_0: Int64
float_col_1: Float64
cat_col_2: Utf8
int_col_3: Int64
float_col_4: Float64
cat_col_5: Utf8
int_col_6: Int64
float_col_7: Float64
cat_col_8: Utf8
Optimized id: Int64 -> UInt16
Optimized int_col_0: Int64 -> Int16
Optimized float_col_1: Float64 -> Float32
String stats: 10/2000 unique values (0.50%)
Converting to categorical: 0.5% <= 50.0%
Optimized cat_col_2: Utf8 -> Categorical
Optimized int_col_3: Int64 -> Int16
Optimized float_col_4: Float64 -> Float32
String stats: 10/2000 unique values (0.50%)
Converting to categorical: 0.5% <= 50.0%
Optimized cat_col_5: Utf8 -> Categorical
Optimized int_col_6: Int64 -> Int16
Optimized float_col_7: Float64 -> Float32
String stats: 10/2000 unique values (0.50%)
Converting to categorical: 0.5% <= 50.0%
Optimized cat_col_8: Utf8 -> Categorical

Original schema:
__row_id: UInt32


## Display Results

Show the comparison report inline:

In [5]:
results.display_html()

Observation,Base Value,Compare Value,Difference,% Difference
31,cat_8,modified,None,None
265,cat_1,modified,None,None
337,cat_8,modified,None,None
443,cat_9,modified,None,None
458,cat_6,modified,None,None
483,cat_5,modified,None,None
711,cat_9,modified,None,None
737,cat_8,modified,None,None
764,cat_0,modified,None,None
1053,cat_0,modified,None,None
